In [1]:
from pathlib import Path

import polars as pl

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent.resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "anomalies"
    / "precipitation_amount.parquet"
)

assert INPUT_FILE.exists()

print("=" * 60)
print("VALIDATE ANOMALIES")
print("=" * 60)
print(f"Input : {INPUT_FILE}")
print("=" * 60)

VALIDATE ANOMALIES
Input : C:\Projects\ews\zoryn\data\processed\anomalies\precipitation_amount.parquet


In [3]:
# ============================================================
# LOAD ANOMALIES
# ============================================================

anomalies = pl.scan_parquet(INPUT_FILE)

summary = (
    anomalies
    .select(
        [
            pl.len().alias("rows"),
            pl.n_unique("h3_id").alias("h3_cells"),
            pl.n_unique("timestamp").alias("timestamps"),
            pl.min("anomaly").alias("min_anomaly"),
            pl.max("anomaly").alias("max_anomaly"),
            pl.min("z_score").alias("min_z"),
            pl.max("z_score").alias("max_z"),
        ]
    )
    .collect()
)

expected_rows = (
    summary["h3_cells"][0]
    * summary["timestamps"][0]
)

print("=" * 60)
print("ANOMALY DATASET")
print("=" * 60)
print(summary)
print()
print(f"Expected Rows : {expected_rows:,}")
print("=" * 60)

assert summary["rows"][0] == expected_rows

ANOMALY DATASET
shape: (1, 7)
┌──────────┬──────────┬────────────┬─────────────┬─────────────┬───────────┬──────────┐
│ rows     ┆ h3_cells ┆ timestamps ┆ min_anomaly ┆ max_anomaly ┆ min_z     ┆ max_z    │
│ ---      ┆ ---      ┆ ---        ┆ ---         ┆ ---         ┆ ---       ┆ ---      │
│ u32      ┆ u32      ┆ u32        ┆ f32         ┆ f32         ┆ f32       ┆ f32      │
╞══════════╪══════════╪════════════╪═════════════╪═════════════╪═══════════╪══════════╡
│ 11679216 ┆ 42316    ┆ 276        ┆ -186.139374 ┆ 418.70636   ┆ -2.204687 ┆ 4.252658 │
└──────────┴──────────┴────────────┴─────────────┴─────────────┴───────────┴──────────┘

Expected Rows : 11,679,216


In [4]:
# ============================================================
# ANOMALY DISTRIBUTION QA
# ============================================================

distribution = (
    anomalies
    .select(
        [
            pl.col("anomaly").mean().alias("mean_anomaly"),
            pl.col("anomaly").std().alias("std_anomaly"),
            pl.col("z_score").mean().alias("mean_z"),
            pl.col("z_score").std().alias("std_z"),
        ]
    )
    .collect()
)

print("=" * 60)
print("ANOMALY DISTRIBUTION")
print("=" * 60)
print(distribution)
print("=" * 60)

ANOMALY DISTRIBUTION
shape: (1, 4)
┌──────────────┬─────────────┬─────────────┬──────────┐
│ mean_anomaly ┆ std_anomaly ┆ mean_z      ┆ std_z    │
│ ---          ┆ ---         ┆ ---         ┆ ---      │
│ f32          ┆ f32         ┆ f32         ┆ f32      │
╞══════════════╪═════════════╪═════════════╪══════════╡
│ -2.7660e-8   ┆ 53.957703   ┆ -3.8282e-10 ┆ 0.978019 │
└──────────────┴─────────────┴─────────────┴──────────┘


In [ ]:
# ============================================================
# NaN QA
# ============================================================

df = anomalies.collect()

print("=" * 60)
print("NaN QA")
print("=" * 60)

print(f"Anomaly NaNs : {df['anomaly'].is_nan().sum()}")
print(f"Z-score NaNs : {df['z_score'].is_nan().sum()}")

print(f"Anomaly Nulls: {df['anomaly'].is_null().sum()}")
print(f"Z-score Nulls: {df['z_score'].is_null().sum()}")

assert df["anomaly"].is_nan().sum() == 0
assert df["z_score"].is_nan().sum() == 0

assert df["anomaly"].is_null().sum() == 0
assert df["z_score"].is_null().sum() == 0

print("QA PASSED")
print("=" * 60)

NaN QA
shape: (1, 4)
┌─────────────┬───────┬──────────────┬────────┐
│ anomaly_nan ┆ z_nan ┆ anomaly_null ┆ z_null │
│ ---         ┆ ---   ┆ ---          ┆ ---    │
│ u32         ┆ u32   ┆ u32          ┆ u32    │
╞═════════════╪═══════╪══════════════╪════════╡
│ 828         ┆ 828   ┆ 0            ┆ 0      │
└─────────────┴───────┴──────────────┴────────┘
